# E01

In [ ]:

from __future__ import annotations

import csv
import json
import math
import sys
from copy import deepcopy
from pathlib import Path


import shutil
import tempfile
import zipfile

SOURCE_ZIP_NAME = 'syniscopy_source.zip'
IN_COLAB = Path('/content').exists()
DRIVE_MYDRIVE = Path('/content/drive/MyDrive')
if IN_COLAB and not DRIVE_MYDRIVE.exists():
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as exc:
        print('Drive was not mounted automatically:', repr(exc))


def find_supplemental_root() -> Path:
    explicit = globals().get('SYNISCOPY_SUPPLEMENTAL_ROOT', None)
    candidates = []
    if explicit:
        candidates.append(Path(explicit).expanduser())
    here = Path.cwd().resolve()
    candidates.extend([here, *here.parents])
    if DRIVE_MYDRIVE.exists():
        candidates.extend([
            DRIVE_MYDRIVE / 'supplemental',
            DRIVE_MYDRIVE / 'SyniscopySupplemental',
        ])
    candidates.extend([
        Path('/content/supplemental'),
        Path('/content/SyniscopySupplemental'),
    ])
    for candidate in candidates:
        try:
            resolved = candidate.resolve()
        except Exception:
            resolved = candidate
        if (resolved / 'supplemental' / 'E01.ipynb').exists():
            resolved = resolved / 'supplemental'
        if (resolved / 'E01.ipynb').exists() and (
            (resolved / SOURCE_ZIP_NAME).exists() or (resolved.parent / 'codebase').is_dir()
        ):
            return resolved
    raise RuntimeError(
        'Syniscopy supplemental folder not found. Upload the supplemental folder to Drive as MyDrive/supplemental, '
        'including syniscopy_source.zip.'
    )


def prepare_syniscopy_source(supplemental_root: Path) -> Path:
    source_root = supplemental_root.parent
    if (source_root / 'codebase').is_dir():
        return source_root
    zip_path = supplemental_root / SOURCE_ZIP_NAME
    if not zip_path.exists():
        raise FileNotFoundError(
            f'Missing {zip_path}. Run python supplemental/package_experiments_for_colab.py before uploading supplemental/.'
        )
    extract_dir = Path('/content/syniscopy_source') if IN_COLAB else Path(tempfile.gettempdir()) / 'syniscopy_source'
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(extract_dir)
    if (extract_dir / 'codebase').is_dir():
        return extract_dir
    candidates = [p for p in extract_dir.iterdir() if p.is_dir() and (p / 'codebase').is_dir()]
    if len(candidates) != 1:
        raise RuntimeError(f'Could not find a single codebase/ root after extracting {zip_path}.')
    return candidates[0]


SUPPLEMENTAL_ROOT = find_supplemental_root()
ROOT = prepare_syniscopy_source(SUPPLEMENTAL_ROOT)
CODEBASE = ROOT / 'codebase'
OUTPUT_DIR = SUPPLEMENTAL_ROOT / 'outputs' / 'E01'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if str(CODEBASE) not in sys.path:
    sys.path.insert(0, str(CODEBASE))

import numpy as np

from config import PARAMS
from fisher_diagnostic import (
    compare_modality_axial_information_content,
    compare_modality_information_content,
    compare_modality_information_content_detected_quanta_normalized,
    compute_localization_crlb,
    compute_modality_fusion_crlb,
)
from imaging_model import modality_display_name
from main import generate_single_frame_views

HEADLINE_MODALITIES = [
    'bright_field',
    'fluorescence_widefield',
    'tirf_fluorescence',
    'dark_field',
    'zernike_phase_contrast',
    'differential_phase_contrast',
    'quantitative_phase',
    'off_axis_holography',
    'ricm',
    'interferometric',
    'tem_phase_contrast',
    'sem_secondary_electron',
]
PANEL_MODALITIES = [
    'bright_field',
    'fluorescence_widefield',
    'tirf_fluorescence',
    'dark_field',
    'zernike_phase_contrast',
    'differential_phase_contrast',
    'quantitative_phase',
    'off_axis_holography',
    'ricm',
    'interferometric',
    'tem_phase_contrast',
    'sem_secondary_electron',
]

IMAGE_SIZE = 192
PUPIL_SAMPLES = 192
PIXEL_SIZE_NM = 65.0
WAVELENGTH_NM = 532.0
DIAMETER_NM = 100.0
Z_NM = 0.0
Z_STEP_NM = 200.0
NOISE_VARIANCE = 1.0
QUANTA_BUDGET = 5e4
BUDGET_READOUT_VARIANCE = 1.0




def finite_or_inf(value: float, digits: int = 3) -> str:
    value = float(value)
    if not np.isfinite(value):
        return '$\\infty$'
    return f'{value:.{digits}g}'


def stable_seed(label: str) -> int:
    seed = 0
    for index, char in enumerate(str(label)):
        seed = (seed + (index + 1) * ord(char)) % (2**32)
    return seed


def display_from_contrast(contrast: np.ndarray, label: str, display_noise_std: float = 0.01) -> np.ndarray:
    image = np.asarray(contrast, dtype=float)
    finite = image[np.isfinite(image)]
    if finite.size == 0:
        return np.zeros(image.shape, dtype=np.uint8)
    center = float(np.median(finite))
    deviation = np.where(np.isfinite(image), image - center, 0.0)
    finite_deviation = deviation[np.isfinite(deviation)]
    scale = float(np.percentile(np.abs(finite_deviation), 99.5)) if finite_deviation.size else 0.0
    if not np.isfinite(scale) or scale <= 0.0:
        scale = float(np.max(np.abs(finite_deviation))) if finite_deviation.size else 0.0
    if not np.isfinite(scale) or scale <= 0.0:
        return np.full(image.shape, 128, dtype=np.uint8)
    normalized = 0.5 + 0.42 * deviation / scale
    if display_noise_std > 0.0:
        rng = np.random.default_rng(stable_seed(label))
        normalized = normalized + rng.normal(0.0, display_noise_std, normalized.shape)
    return np.clip(255.0 * normalized, 0.0, 255.0).astype(np.uint8)


def particle(diameter_nm: float, z_nm: float, material: str = 'polystyrene') -> dict:
    material_properties = None
    if material == 'fluorescent_polystyrene':
        material_properties = {
            'fluorophore_density': 0.08,
            'excitation_peak_nm': 488.0,
            'emission_peak_nm': 520.0,
        }
    return {
        'name': f'{material}_{diameter_nm:g}nm',
        'motion': {'hydrodynamic_diameter_nm': float(diameter_nm), 'initial_position_nm': None},
        'signal_multiplier': 1.0,
        'components': [{
            'shape': 'sphere',
            'offset_nm': [0.0, 0.0, 0.0],
            'diameter_nm': float(diameter_nm),
            'material': material,
            'refractive_index': None,
            'signal_multiplier': 1.0,
            'material_properties': material_properties,
        }],
    }


def base_params(modality: str, diameter_nm: float, z_nm: float) -> dict:
    material = 'fluorescent_polystyrene' if 'fluorescence' in modality else 'polystyrene'
    params = deepcopy(PARAMS)
    params.update({
        'imaging_model': modality,
        'image_size_pixels': IMAGE_SIZE,
        'pixel_size_nm': PIXEL_SIZE_NM,
        'pupil_samples': PUPIL_SAMPLES,
        'psf_oversampling_factor': 2,
        'fps': 24.0,
        'duration_seconds': 1.0 / 24.0,
        'wavelength_nm': WAVELENGTH_NM,
        'numerical_aperture': 1.0,
        'refractive_index_medium': 1.33,
        'refractive_index_immersion': 1.518,
        'background_intensity': 1000.0,
        'read_noise_counts': 1.0,
        'camera_gain_e_per_count': 1.0,
        'shot_noise_enabled': False,
        'gaussian_noise_enabled': False,
        'fixed_pattern_gain_std': 0.0,
        'fixed_pattern_offset_counts': 0.0,
        'hot_pixel_fraction': 0.0,
        'scan_line_noise_counts': 0.0,
        'return_ideal_float_frames': True,
        'save_frame_sequence': False,
        'save_raw_frame_views': False,
        'mask_generation_enabled': False,
        'sample_environment_enabled': False,
        'sample_environment_pattern_enabled': False,
        'sample_environment_pattern': 'none',
        'background_subtraction_method': 'reference_frame',
        'z_stack_step_nm': 100.0,
        'initial_z_span_nm': max(4000.0, abs(float(z_nm)) * 2.0 + 1000.0),
        'particles': [particle(float(diameter_nm), float(z_nm), material=material)],
        'channels': None,
    })
    side_nm = IMAGE_SIZE * PIXEL_SIZE_NM
    params['particles'][0]['motion']['initial_position_nm'] = [0.5 * side_nm, 0.5 * side_nm, float(z_nm)]
    return params


def render(modality: str, diameter_nm: float = DIAMETER_NM, z_nm: float = Z_NM):
    params = base_params(modality, diameter_nm, z_nm)
    views = generate_single_frame_views(params)
    contrast = views.get('contrast_frame')
    signal_counts = views.get('ideal_signal_frame')
    if signal_counts is None:
        signal_counts = views.get('raw_signal_frame')
    if contrast is None:
        raise RuntimeError(f'{modality} did not produce a contrast frame.')
    if signal_counts is None:
        raise RuntimeError(f'{modality} did not produce a detector-count frame.')
    contrast = np.asarray(contrast, dtype=float)
    final = display_from_contrast(contrast, f'{modality}:{diameter_nm}:{z_nm}')
    return contrast, final, np.asarray(signal_counts, dtype=float)


from PIL import Image


def save_uint8_png(path: Path, image: np.ndarray) -> None:
    arr = np.asarray(image, dtype=np.uint8)
    Image.fromarray(arr).save(path)


image_dir = OUTPUT_DIR / 'modality_images'
array_dir = OUTPUT_DIR / 'modality_arrays'
image_dir.mkdir(parents=True, exist_ok=True)
array_dir.mkdir(parents=True, exist_ok=True)

manifest_rows = []
for modality in PANEL_MODALITIES:
    print('render', modality)
    contrast, image, counts = render(modality)
    image_path = image_dir / f'{modality}.png'
    contrast_path = array_dir / f'{modality}_contrast.npy'
    counts_path = array_dir / f'{modality}_detector_counts.npy'
    save_uint8_png(image_path, image)
    np.save(contrast_path, contrast)
    np.save(counts_path, counts)
    manifest_rows.append({
        'modality': modality,
        'display_name': modality_display_name(modality),
        'image_png': str(image_path.relative_to(OUTPUT_DIR)),
        'contrast_npy': str(contrast_path.relative_to(OUTPUT_DIR)),
        'detector_counts_npy': str(counts_path.relative_to(OUTPUT_DIR)),
        'contrast_min': float(np.nanmin(contrast)),
        'contrast_max': float(np.nanmax(contrast)),
        'detector_counts_sum': float(np.nansum(counts)),
    })

manifest_csv = OUTPUT_DIR / 'modality_manifest.csv'
with manifest_csv.open('w', newline='', encoding='utf-8') as fh:
    fieldnames = list(manifest_rows[0]) if manifest_rows else []
    writer = csv.DictWriter(fh, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(manifest_rows)
manifest_json = OUTPUT_DIR / 'modality_manifest.json'
manifest_json.write_text(json.dumps(manifest_rows, indent=2, allow_nan=False), encoding='utf-8')
print('wrote raw E01 modality artifacts:', OUTPUT_DIR)
